## FarU project

Concept demonstration:

Synthetic NER dataset -> token classification (LESION, SIZE, LOCATION, LATERALITY) -> LoRA -> train+eval

We develope code with the support of the AI co-pilot

In [ ]:
import os
import random
from typing import List, Dict, Any
import math

import torch
import numpy as np

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
# CONFIG

BASE_MODEL = "mistralai/Mistral-Small-24B-Instruct-2501"  # change to your base model
OUTPUT_DIR = "./mistral_lora_medical_ner"
MAX_LENGTH = 128
SEED = 42
NUM_SAMPLES = 4000  # synthetic dataset size (balanced)
TRAIN_TEST_SPLIT = 0.1
BATCH_SIZE = 4
EPOCHS = 3
LR = 5e-5

os.environ["WANDB_DISABLED"] = "true"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"


# LABELS (BIO scheme)
label_list = [
    "O",
    "B-LESION", "I-LESION",
    "B-SIZE", "I-SIZE",
    "B-LOCATION", "I-LOCATION",
    "B-LATERALITY", "I-LATERALITY"
]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}


# SYNTHETIC DATA GENERATOR

lesions = ["nodule", "mass", "cyst", "lesion", "opacity", "tumor", "growth"]
sizes = ["3 mm", "5 mm", "7 mm", "10 mm", "12 mm", "15 mm", "subcentimeter", "tiny", "small"]
locations = [
    "right upper lobe", "left upper lobe", "right lower lobe", "left lower lobe",
    "right middle lobe", "lingula", "left lingula", "bilateral lungs", "apical segment", "basal segment"
]
laterality_terms = {
    "right": ["right", "RUL", "RLL", "right-sided"],
    "left": ["left", "LUL", "LLL", "left-sided"],
    "bilateral": ["bilateral", "both lungs"]
}

neg_templates = [
    "Lungs are clear.",
    "No pulmonary nodules or suspicious lesions identified.",
    "No focal lung abnormalities.",
    "Exam without acute cardiopulmonary disease."
]

positive_templates = [
    "{size} {lesion} in the {location}.",
    "{lesion} measuring {size} located in the {location}.",
    "There is a {size} {lesion} in the {location}.",
    "{lesion} of about {size} seen in the {location}.",
    "Impression: {size} {lesion} at the {location}."
]



In [ ]:
def choose_laterality_for_location(location: str):
    # infer laterality from location (simple heuristic)
    if "right" in location:
        return "right"
    if "left" in location:
        return "left"
    if "bilateral" in location:
        return "bilateral"
    # otherwise random
    return random.choice(["right", "left", "bilateral"])

def synth_example_positive():
    lesion = random.choice(lesions)
    size = random.choice(sizes)
    location = random.choice(locations)
    laterality = choose_laterality_for_location(location)
    lat_term = random.choice(laterality_terms[laterality])
    template = random.choice(positive_templates)
    text = template.format(size=size, lesion=lesion, location=location)
    # sometimes prepend laterality phrase
    if random.random() < 0.3:
        text = f"{lat_term} {text}"
    # sometimes append follow-up phrase
    if random.random() < 0.15:
        text += " Recommend follow-up imaging."
    return text, lesion, size, location, laterality

def synth_example_negative():
    return random.choice(neg_templates), None, None, None, None

# -------------------
# Convert sentence -> tokens + BIO labels (word-level), then we'll align to subwords later
# -------------------
def simple_tokenize_words(text: str) -> List[str]:
    # simple whitespace tokenizer; keep punctuation attached (works with alignment)
    return text.strip().split()

def label_tokens_for_entities(words: List[str], lesion: str, size: str, location: str, laterality: str):
    labels = ["O"] * len(words)
    lower_words = [w.lower().strip(".,;:") for w in words]

    # mark size (could be multiple tokens)
    if size:
        size_tokens = size.split()
        # naive search for size sequence
        for i in range(len(lower_words) - len(size_tokens) + 1):
            if lower_words[i:i+len(size_tokens)] == [t.lower() for t in size_tokens]:
                labels[i] = "B-SIZE"
                for j in range(1, len(size_tokens)):
                    labels[i+j] = "I-SIZE"
                break

    # mark lesion (single token usually)
    if lesion:
        for i, w in enumerate(lower_words):
            # exact match or punctuation-trim match
            if w == lesion.lower():
                labels[i] = "B-LESION"
                # no multi-token lesion in our generator, but keep consistent
                # check next tokens if they are part of lesion phrase (rare)
                if i+1 < len(lower_words) and lower_words[i+1].startswith("lesion"):
                    labels[i+1] = "I-LESION"
                break

    # mark location (multiword)
    if location:
        loc_tokens = [t.lower() for t in location.split()]
        for i in range(len(lower_words) - len(loc_tokens) + 1):
            if lower_words[i:i+len(loc_tokens)] == loc_tokens:
                labels[i] = "B-LOCATION"
                for j in range(1, len(loc_tokens)):
                    labels[i+j] = "I-LOCATION"
                break

    # mark laterality (right/left/bilateral) - find any laterality token
    if laterality:
        lat_options = laterality_terms[laterality]
        for i, w in enumerate(lower_words):
            if w in [opt.lower().strip(".,") for opt in lat_options]:
                labels[i] = "B-LATERALITY"
                # if token like "both" -> probably single token
                break

    return labels

In [ ]:


# Build synthetic dataset
examples = []
num_pos = NUM_SAMPLES // 2
num_neg = NUM_SAMPLES - num_pos

for _ in range(num_pos):
    text, lesion, size, location, laterality = synth_example_positive()
    words = simple_tokenize_words(text)
    ner_tags = label_tokens_for_entities(words, lesion, size, location, laterality)
    examples.append({"tokens": words, "ner_tags": [label2id[t] for t in ner_tags], "text": text})

for _ in range(num_neg):
    text, lesion, size, location, laterality = synth_example_negative()
    words = simple_tokenize_words(text)
    ner_tags = ["O"] * len(words)
    examples.append({"tokens": words, "ner_tags": [label2id[t] for t in ner_tags], "text": text})

random.shuffle(examples)

# Convert to HuggingFace Dataset and split
ds = Dataset.from_list(examples)
split = ds.train_test_split(test_size=TRAIN_TEST_SPLIT, seed=SEED)
dataset = DatasetDict({"train": split["train"], "test": split["test"]})

print("Dataset sizes:", {k: len(v) for k, v in dataset.items()})

# -------------------
# TOKENIZER + ALIGNMENT
# -------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.padding_side = "right"

def tokenize_and_align_labels(batch: Dict[str, Any]):
    # batch["tokens"] is a list of token lists
    tokenized_inputs = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

    all_labels = []
    for i, labels in enumerate(batch["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            else:
                label = labels[word_idx]
                # For B/I logic we keep the same label on first word, I-... on subsequent subwords.
                aligned_labels.append(label)
        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

# Map dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True, remove_columns=dataset["train"].column_names)
tokenized_datasets.set_format(type="torch")

# -------------------
# MODEL (QLoRA + LoRA)
# -------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Resize token embeddings if we added pad token
model.resize_token_embeddings(len(tokenizer))

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# -------------------
# METRICS
# -------------------
# seqeval expects lists of entity labels (strings) per sentence
def align_predictions(predictions: np.ndarray, label_ids: np.ndarray):
    preds = np.argmax(predictions, axis=-1)
    batch_size, seq_len = preds.shape

    out_preds = []
    out_labels = []

    for i in range(batch_size):
        example_preds = []
        example_labels = []
        for j in range(seq_len):
            if label_ids[i, j] == -100:
                continue
            example_preds.append(label_list[preds[i, j]])
            example_labels.append(label_list[label_ids[i, j]])
        out_preds.append(example_preds)
        out_labels.append(example_labels)
    return out_preds, out_labels

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    preds, refs = align_predictions(logits, label_ids)
    p = precision_score(refs, preds)
    r = recall_score(refs, preds)
    f1 = f1_score(refs, preds)
    return {"precision": p, "recall": r, "f1": f1}

# -------------------
# TRAINING
# -------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=False,
    bf16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=default_data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save LoRA adapter + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter & tokenizer to", OUTPUT_DIR)

# -------------------
# INFERENCE (NER extraction)
# -------------------
model.eval()

def ner_predict(text):
    # Split text into words exactly like during dataset creation
    words = text.split()

    # Tokenize in "is_split_into_words" mode (same as training)
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        logits = model(**enc).logits

    predictions = torch.argmax(logits, dim=-1)[0].cpu().numpy()
    word_ids = enc.word_ids(batch_index=0)

    # Reconstruct NER labels aligned to words
    results = []
    current_word_idx = None
    current_word_label = None

    for pred_label_id, word_id in zip(predictions, word_ids):
        if word_id is None:
            continue  # skip special tokens

        label = id2label[pred_label_id]

        # New word begins
        if word_id != current_word_idx:
            # Append previous word result
            if current_word_label is not None:
                results.append(current_word_label)

            current_word_idx = word_id
            current_word_label = (words[word_id], label)
        else:
            # For subword tokens, use I- prefix if needed
            if label.startswith("I-"):
                current_word_label = (current_word_label[0], label)

    # append last
    if current_word_label is not None:
        results.append(current_word_label)

    return results





# Example inference
print("Example inference:")
ex_sentence = "Right 7 mm nodule in the right upper lobe noted on CT."
ents = ner_predict(ex_sentence)
print("Text:", ex_sentence)
print("Entities:", ents)